In [7]:
from langchain.tools import tool

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from pydantic import BaseModel, Field

#from fastmcp import FastMCP
#import sqlalchemy as sa
#import pandas as pd

#from utils.logger import get_logger
#_logs = get_logger(__name__)

from dotenv import load_dotenv
import os


load_dotenv()
load_dotenv(".secrets")

vector_db_client_url="http://localhost:8000"
chroma = chromadb.HttpClient(host=vector_db_client_url)
collection = chroma.get_collection(name="pitchfork_reviews", 
                                   embedding_function=OpenAIEmbeddingFunction(
                                       api_key = os.getenv("OPENAI_API_KEY"),
                                       model_name="text-embedding-3-small")
                                   )


class MusicReviewData(BaseModel):
    """Structured music review data response."""
    title: str = Field(..., description="The title of the album.")
    artist: str = Field(..., description="The artist of the album.")
    review: str = Field(..., description="A portion of the album review that is relevant to the user query.")
    score: float = Field(None, description="The Pitchfork score of the album. The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.")


@tool
def semantic_search(query: str, n_results: int = 1) -> list["MusicReviewData"]:
    """Fetches music review data based on the query. Returns n_results reviews."""

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
# def recommend_albums(query: str, n_results: int = 1) -> list[MusicReviewData]:
#     """Fetches music review data based on the query. Returns n_results reviews."""
#     recommendations = get_context(query, collection, n_results)
#     return recommendations


# def additional_details(review_id:str):
#     _logs.debug(f'Fetching additional details for review ID: {review_id}')
#     engine = sa.create_engine(os.getenv("SQL_URL"))
#     query = f"""
#     SELECT r.reviewid,
# 		r.title,
# 		r.artist,
# 		r.score,
# 		g.genre
#     FROM reviews AS r
#     LEFT JOIN genres as g
# 	    ON r.reviewid = g.reviewid
#     WHERE r.reviewid = '{review_id}'
#     """
#     with engine.connect() as conn:
#         result = pd.read_sql(query, conn)
#     if not result.empty:
#         row = result.iloc[0]
#         details = {
#             "reviewid": row['reviewid'],
#             "album": row['title'],
#             "score": row['score'],
#             "artist": row['artist']
#         }
#         return details
#     else:
#         _logs.warning(f'No details found for review ID: {review_id}')
#         return {}
    
# def get_reviewid_from_custom_id(custom_id:str):
#     return custom_id.split('_')[0]

# def get_context_data(query:str, collection:chromadb.api.models.Collection, top_n:int):
#     results = collection.query(
#         query_texts=[query],
#         n_results=top_n
#     )
#     context_data = []
#     for idx, custom_id in enumerate(results['ids'][0]):
#         review_id = get_reviewid_from_custom_id(custom_id)
#         details = additional_details(review_id)
#         details['text'] = results['documents'][0][idx]
#         context_data.append(details)
#     return context_data

# def get_context(query:str, collection:chromadb.api.models.Collection, top_n:int):
#     context_data = get_context_data(query, collection, top_n)
#     recommendations = []
#     if not context_data:
#         return recommendations
#     for item in context_data:

#         rec = MusicReviewData(
#             title=item.get('album', 'N/A'),
#             artist=item.get('artist', 'N/A'),
#             review=item.get('text', 'N/A'),
#             score=item.get('score', 0.0)
#         )
#         recommendations.append(rec)
#     return recommendations


In [13]:
import sys
print(sys.path)

['c:\\Users\\atwalgag\\Documents\\deploying-ai\\05_src\\assignment_chat', 'C:\\Users\\atwalgag\\AppData\\Roaming\\uv\\python\\cpython-3.12.13-windows-x86_64-none\\python312.zip', 'C:\\Users\\atwalgag\\AppData\\Roaming\\uv\\python\\cpython-3.12.13-windows-x86_64-none\\DLLs', 'C:\\Users\\atwalgag\\AppData\\Roaming\\uv\\python\\cpython-3.12.13-windows-x86_64-none\\Lib', 'C:\\Users\\atwalgag\\AppData\\Roaming\\uv\\python\\cpython-3.12.13-windows-x86_64-none', 'c:\\Users\\atwalgag\\Documents\\deploying-ai\\deploying-ai-env', '', 'c:\\Users\\atwalgag\\Documents\\deploying-ai\\deploying-ai-env\\Lib\\site-packages', 'c:\\Users\\atwalgag\\Documents\\deploying-ai\\deploying-ai-env\\Lib\\site-packages\\win32', 'c:\\Users\\atwalgag\\Documents\\deploying-ai\\deploying-ai-env\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\atwalgag\\Documents\\deploying-ai\\deploying-ai-env\\Lib\\site-packages\\Pythonwin']


In [15]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START
from langchain.chat_models import init_chat_model
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from langchain_core.messages import SystemMessage,  HumanMessage

import gradio as gr

from dotenv import load_dotenv
import json
import requests
import os

load_dotenv('../../05_src/.secrets')
load_dotenv('../../05_src/.env')

import os.path
import sys
sys.path.append(os.path.join(os.path.dirname(assignment), '../../05_src/'))

from assignment_chat.prompts import return_instructions
from assignment_chat.tools_search import semantic_search

#Gradio User interface
# Attributed to https://www.gradio.app/guides/chatinterface-examples#lang-chain

import gradio as gr
from langchain.messages import AIMessage, HumanMessage  
from langchain_openai import ChatOpenAI  

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)
instructions = return_instructions()
tools = [semantic_search]

def predict(message, history):
    history_langchain_format = []
    for msg in history:
        if msg["role"] == "user":
            history_langchain_format.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            history_langchain_format.append(AIMessage(content=msg["content"]))
    history_langchain_format.append(HumanMessage(content=message))
    gpt_response = model.invoke(history_langchain_format)
    return gpt_response.content


chatInterface = gr.ChatInterface(
    predict,
    api_name="chat",
)

#Will open at a localhost instance e.g., http://127.0.0.1:7860
chatInterface.launch()


NameError: name '__file__' is not defined